In [1]:
# ============================================
# Cell 1: 下载 Kaggle 数据集 + 配置数据路径
# ============================================
import os, shutil

# 清除可能损坏的旧缓存
cache_dir = os.path.expanduser("~/.cache/kagglehub")
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

# 安装 kagglehub
!pip install kagglehub -q

import kagglehub

# 下载数据集
print("正在下载数据集（约 2.97GB，需 5-10 分钟）...")
kaggle_path = kagglehub.dataset_download(
    "patrickfleith/spacecraft-thruster-firing-tests-dataset",
    force_download=True
)
print(f"✅ 下载完成: {kaggle_path}")

# ========== 👇 修复点：先创建目录再进入 ==========
project_dir = "/home/ma-user/work/thruster_project"
os.makedirs(project_dir, exist_ok=True)   # 创建目录（如果已存在则不报错）
os.chdir(project_dir)                     # 再进入该目录
# =================================================

os.makedirs("data/dataset/dataset", exist_ok=True)

# 清除旧链接
for link in ["data/dataset/dataset/train", "data/metadata.csv"]:
    if os.path.islink(link):
        os.unlink(link)     # 注意：这里是 unlink，不是 "1ink"

# 创建新链接
os.symlink(
    os.path.join(kaggle_path, "dataset", "dataset", "train"),
    "data/dataset/dataset/train"
)
os.symlink(
    os.path.join(kaggle_path, "metadata.csv"),
    "data/metadata.csv"
)

# 验证
import pandas as pd
df_meta = pd.read_csv("data/metadata.csv")
print(f"✅ metadata 行数: {len(df_meta)}")
print(f"✅ 训练文件数: {len(os.listdir('data/dataset/dataset/train'))}")
print(f"✅ 数据配置完成，可以进入 Cell 2")


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


/home/ma-user/anaconda3/envs/PyTorch-2.7.1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


正在下载数据集（约 2.97GB，需 5-10 分钟）...


100%|██████████| 2.97G/2.97G [06:12<00:00, 8.59MB/s] 

Extracting files...


✅ 下载完成: /home/ma-user/.cache/kagglehub/datasets/patrickfleith/spacecraft-thruster-firing-tests-dataset/versions/1
✅ metadata 行数: 2612
✅ 训练文件数: 1267
✅ 数据配置完成，可以进入 Cell 2


In [2]:
%%writefile model_train_v2_resume.py
"""完整训练脚本 — 支持中断续跑"""
import os, torch, sys, time
sys.path.insert(0, ".")

if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch, 'npu') and torch.npu.is_available():
    device = torch.device('npu')
else:
    device = torch.device('cpu')
print(f"Device: {device}")

from data_pipeline_v2 import INPUT_DIM, save_meta_info
from model_train_v2 import ThrusterDataset, DualOutputLSTM, dual_loss
from torch.utils.data import DataLoader
import torch.optim as optim
import pandas as pd
import numpy as np

metadata_path = "data/metadata.csv"
train_dir = "data/dataset/dataset/train/"

df_all = pd.read_csv(metadata_path)
df_all['filename'] = df_all['filename'].str.strip()
df_all = df_all[df_all['anomalous'] == False]
all_files = df_all['filename'].tolist()
np.random.seed(42)
np.random.shuffle(all_files)
split = int(len(all_files) * 0.8)
train_files = set(all_files[:split])
val_files = set(all_files[split:])
print(f"Train files: {len(train_files)}, Val files: {len(val_files)}")

print("Building train dataset...")
t0 = time.time()
train_dataset = ThrusterDataset(metadata_path, train_dir, seq_len=200, stride=100, file_filter=train_files)
print(f"  Done ({time.time()-t0:.0f}s) — {len(train_dataset)} windows")

print("Building val dataset...")
t0 = time.time()
val_dataset = ThrusterDataset(metadata_path, train_dir, seq_len=200, stride=200, file_filter=val_files)
print(f"  Done ({time.time()-t0:.0f}s) — {len(val_dataset)} windows")

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

model = DualOutputLSTM(input_dim=INPUT_DIM, hidden_dim=256, num_layers=2, n_heads=4).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=2e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-5)

save_dir = "outputs/models/v2"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, "dual_output_lstm_v2.pth")
checkpoint_path = os.path.join(save_dir, "checkpoint.pt")

epochs = 100
start_epoch = 0
best_val_loss = float("inf")
early_stop_patience = 20
epochs_no_improve = 0

if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch = ckpt['epoch']
    best_val_loss = ckpt['best_val_loss']
    epochs_no_improve = ckpt.get('epochs_no_improve', 0)
    print(f"  >> 续跑: epoch {start_epoch}/{epochs}, best_val_loss={best_val_loss:.6f}")
elif os.path.exists(save_path):
    model.load_state_dict(torch.load(save_path, map_location=device))
    print(f"  >> 热启动: 加载权重 ({os.path.getsize(save_path)/1024:.0f} KB)")
else:
    print("  >> 从头训练")

print(f"\nTraining epochs {start_epoch+1} to {epochs}...\n")

for epoch in range(start_epoch, epochs):
    epoch_t0 = time.time()
    model.train()
    train_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        pred = model(x)
        loss = dual_loss(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            val_loss += dual_loss(pred, y).item()

    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    scheduler.step(val_loss)
    elapsed = time.time() - epoch_t0

    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss
        torch.save(model.state_dict(), save_path)
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    torch.save({
        'epoch': epoch + 1, 'model': model.state_dict(),
        'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
        'best_val_loss': best_val_loss, 'epochs_no_improve': epochs_no_improve,
    }, checkpoint_path)

    marker = " *" if is_best else ""
    print(f"Epoch [{epoch+1:3d}/{epochs}]  {elapsed:.0f}s  train={train_loss:.6f}  val={val_loss:.6f}{marker}")

    if epochs_no_improve >= early_stop_patience:
        print(f"\n>> Early stopping at epoch {epoch+1}")
        break

save_meta_info()
print(f"\n{'='*50}")
print(f"Done! Best val_loss: {best_val_loss:.6f}")
print(f"Model saved: {save_path}")

Writing model_train_v2_resume.py


In [6]:
import os, shutil

os.chdir("/home/ma-user/work")

# 如果旧目录存在，先删掉
if os.path.exists("thruster_project"):
    shutil.rmtree("thruster_project")

# 克隆仓库（包含所有 .py 文件）
!git clone https://github.com/aalandbartelt-cyber/thruster_project.git

os.chdir("/home/ma-user/work/thruster_project")

# 确认文件都到了
print(f"Python 文件: {[f for f in os.listdir('.') if f.endswith('.py')]}")

Cloning into 'thruster_project'...
remote: Enumerating objects: 449, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 449 (delta 15), reused 27 (delta 13), pack-reused 418 (from 1)
Receiving objects: 100% (449/449), 36.14 MiB | 695.00 KiB/s, done.
Resolving deltas: 100% (203/203), done.
Python 文件: ['data_pipeline_v2.py']


In [7]:
import os
os.chdir("/home/ma-user/work/thruster_project")

# 切换到 dev-m 分支（有 model_train_v2.py）
!git checkout dev-m

# 确认文件
print(f"Python 文件: {[f for f in os.listdir('.') if f.endswith('.py')]}")

Branch 'dev-m' set up to track remote branch 'dev-m' from 'origin'.
Switched to a new branch 'dev-m'
Python 文件: ['analyze_ssf.py', 'app_v2.py', 'data_pipeline_v2.py', 'model_shap_v2.py', 'model_finetune_v2.py', 'model_train_v2.py', 'model_test_v2.py', 'inference_utils.py']


In [8]:
import os, shutil, re, pandas as pd, kagglehub

os.chdir("/home/ma-user/work/thruster_project")

# 找 Kaggle 缓存（之前下过，不用重新下）
cache_base = os.path.expanduser("~/.cache/kagglehub/datasets/patrickfleith/spacecraft-thruster-firing-tests-dataset/versions/1")
if not os.path.exists(cache_base):
    # 缓存没了，重新下载
    kp = kagglehub.dataset_download("patrickfleith/spacecraft-thruster-firing-tests-dataset", force_download=True)
else:
    kp = cache_base

print(f"数据路径: {kp}")

# 重建软链接
os.makedirs("data/dataset/dataset", exist_ok=True)
for ln in ["data/dataset/dataset/train", "data/metadata.csv"]:
    if os.path.islink(ln):
        os.unlink(ln)

os.symlink(os.path.join(kp, "dataset", "dataset", "train"), "data/dataset/dataset/train")

# 生成 metadata
train_dir = "data/dataset/dataset/train"
rows = []
for fname in os.listdir(train_dir):
    if not fname.endswith('.csv'):
        continue
    m = re.match(r"(\d+)_(\d+)_SN(\d+)_(\d+)bars_(.+)\.csv", fname)
    if not m:
        continue
    try:
        dfr = pd.read_csv(os.path.join(train_dir, fname))
        last = dfr.iloc[-1]
        ct = last.get('cumulated_throughput', 0)
        co = last.get('cumulated_on_time', 0)
        cp = last.get('cumulated_pulses', 0)
    except:
        ct, co, cp = 0, 0, 0
    rows.append({
        'uid': int(m.group(1)), 'filename': fname,
        'test_id': int(m.group(2)), 'sn': int(m.group(3)),
        'test_pressure': float(m.group(4)), 'test_mode': m.group(5),
        'vl': True, 'anomalous': False, 'anomaly_code': 0,
        'cumulated_throughput': ct, 'cumulated_on_time': co,
        'cumulated_pulses': cp,
    })

pd.DataFrame(rows).to_csv("data/metadata.csv", index=False)
print(f"✅ metadata: {len(rows)} 条, 文件: {len(os.listdir(train_dir))} 个")

# 创建训练脚本
with open("model_train_v2.py", "r") as f:
    code = f.read()

code = code.replace("stride=100", "stride=500")
code = code.replace("stride=200", "stride=1000")

code = code.replace(
    'print(f"\\nTraining {epochs} epochs... (early stop patience={early_stop_patience})")',
    '''import time as _t
ckpt_path = os.path.join(save_dir, "checkpoint.pt")
if os.path.exists(ckpt_path):
    import torch as _th
    ck = _th.load(ckpt_path, map_location=device)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    start_epoch = ck["epoch"]
    best_val_loss = ck["best_val_loss"]
    epochs_no_improve = ck.get("epochs_no_improve", 0)
    print(f"  >> 续跑: epoch {start_epoch}/{epochs}, best={best_val_loss:.6f}")
elif os.path.exists(save_path):
    model.load_state_dict(torch.load(save_path, map_location=device))
    sz = os.path.getsize(save_path) / 1024
    print(f"  >> 热启动: {sz:.0f} KB")
else:
    print("  >> 从头训练")
print(f"\\nTraining epochs {start_epoch+1} to {epochs}...\\n")
for epoch in range(start_epoch, epochs):
    torch.save({"epoch":epoch+1,"model":model.state_dict(),"optimizer":optimizer.state_dict(),"scheduler":scheduler.state_dict(),"best_val_loss":best_val_loss,"epochs_no_improve":epochs_no_improve}, ckpt_path)'''
)
code = code.replace("epochs = 100", "epochs = 100\nstart_epoch = 0")

with open("train.py", "w") as f:
    f.write(code)
print("✅ train.py 已创建")

数据路径: /home/ma-user/.cache/kagglehub/datasets/patrickfleith/spacecraft-thruster-firing-tests-dataset/versions/1
✅ metadata: 1266 条, 文件: 1267 个
✅ train.py 已创建


In [12]:
import os
os.chdir("/home/ma-user/work/thruster_project")

# 先确认 model_train_v2.py 是否存在
if os.path.exists("model_train_v2.py"):
    print("✅ model_train_v2.py 存在")
    # 直接复制一份改 stride
    with open("model_train_v2.py", "r") as f:
        code = f.read()
    code = code.replace("stride=100", "stride=500")
    code = code.replace("stride=200", "stride=1000")
    with open("train.py", "w") as f:
        f.write(code)
    print("✅ train.py 已创建")
else:
    print("❌ model_train_v2.py 不存在，检查分支")
    !git status

✅ model_train_v2.py 存在
✅ train.py 已创建


In [13]:
import os
os.chdir("/home/ma-user/work/thruster_project")
!python -u train.py

/home/ma-user/anaconda3/envs/PyTorch-2.7.1/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/latest owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/home/ma-user/anaconda3/envs/PyTorch-2.7.1/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/8.3.RC1/aarch64-linux/ascend_toolkit_install.info owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/home/ma-user/anaconda3/envs/PyTorch-2.7.1/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/latest owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/home/ma-user/anaconda3/envs/PyTorch-2.7.1/lib/python3.11/site-packages/torch_

In [14]:
import os
os.chdir("/home/ma-user/work/thruster_project")

# 读取 train.py
with open("train.py", "r") as f:
    code = f.read()

# 替换设备检测逻辑
old = "device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')"
new = """if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch, 'npu') and torch.npu.is_available():
    device = torch.device('npu')
else:
    device = torch.device('cpu')"""

code = code.replace(old, new)

with open("train.py", "w") as f:
    f.write(code)

print("✅ NPU 检测已修复")

✅ NPU 检测已修复


In [17]:
import os
os.chdir("/home/ma-user/work/thruster_project")

with open("train.py", "r") as f:
    lines = f.readlines()

# 找到 device 相关的行，打印上下文
print("===== device 相关行 =====")
for i, line in enumerate(lines):
    if 'device' in line.lower() and ('torch' in line or 'cuda' in line or 'npu' in line):
        print(f"L{i+1}: {repr(line)}")

===== device 相关行 =====
L99: "        device = torch.device('cuda')\n"
L101: "        device = torch.device('npu')\n"
L103: "        device = torch.device('cpu')\n"
L196: '    model.load_state_dict(torch.load(save_path, map_location=device))\n'


In [18]:
import os
os.chdir("/home/ma-user/work/thruster_project")

# 读取当前文件
with open("train.py", "r") as f:
    content = f.read()

# 找到并删除所有被搞乱的 device 字符串碎片
import re
# 删除从 L99 到 L103 那些乱掉的 device 行
# 匹配多行字符串中的 device 赋值
content = re.sub(
    r'"""if torch\.cuda\.is_available\(\):.*?device = torch\.device\(\'cpu\'\)"""',
    '',
    content,
    flags=re.DOTALL
)

# 现在写入正确的 device 检测（要匹配 main 函数内的缩进）
content = content.replace(
    "device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')",
    """if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch, 'npu') and torch.npu.is_available():
    device = torch.device('npu')
else:
    device = torch.device('cpu')"""
)

with open("train.py", "w") as f:
    f.write(content)

# 验证
import py_compile
try:
    py_compile.compile("train.py", doraise=True)
    print("✅ 修复成功，语法正确")
except py_compile.PyCompileError as e:
    print(f"❌ 还有错误: {e}")
    # 如果还有错，回退到最简单方案
    print("尝试备用方案...")

❌ 还有错误:   File "train.py", line 104
    elif hasattr(torch, 'npu') and torch.npu.is_available():
    ^^^^
SyntaxError: invalid syntax

尝试备用方案...


In [24]:
import os
os.chdir("/home/ma-user/work/thruster_project")

# 清掉所有乱掉的版本
import shutil
for f in ["train.py", "train.py.bak"]:
    if os.path.exists(f):
        os.remove(f)

# 从原始文件重新复制
!cp model_train_v2.py train.py

# 改 stride
!sed -i 's/stride=100/stride=500/g' train.py
!sed -i 's/stride=200/stride=1000/g' train.py

# 用最小修改方案——只改 device 这一行的检测条件
!sed -i "s/torch.cuda.is_available()/(torch.cuda.is_available() or (hasattr(torch, 'npu') and torch.npu.is_available()))/" train.py

# 再加一行，把 cuda 重定向到 npu
!sed -i "/^device = torch/a\\
import torch;\\
if hasattr(torch, 'npu') and torch.npu.is_available():\\
    torch.cuda.is_available = lambda: False\\
    torch.cuda.device_count = lambda: torch.npu.device_count()\\
    torch.cuda.get_device_name = lambda i: torch.npu.get_device_name(i)\\
    torch.nn.Module.cuda = lambda self, device=None: self.to(device if device else 'npu')" train.py

# 验证
import py_compile
try:
    py_compile.compile("train.py", doraise=True)
    print("✅ 语法正确")
except py_compile.PyCompileError as e:
    print(f"sed 又出问题了，换终极方案")

✅ 语法正确


In [26]:
import os
os.chdir("/home/ma-user/work/thruster_project")

# 重新从原始文件生成
!cp model_train_v2.py train.py
!sed -i 's/stride=100/stride=500/g' train.py
!sed -i 's/stride=200/stride=1000/g' train.py

# 用 Python 改 device 行，保证缩进
with open("train.py", "r") as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    if "device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')" in line:
        new_lines.append("    # 设备检测（支持 NPU）\n")
        new_lines.append("    if torch.cuda.is_available():\n")
        new_lines.append("        device = torch.device('cuda')\n")
        new_lines.append("    elif hasattr(torch, 'npu') and torch.npu.is_available():\n")
        new_lines.append("        device = torch.device('npu')\n")
        new_lines.append("    else:\n")
        new_lines.append("        device = torch.device('cpu')\n")
    else:
        new_lines.append(line)

with open("train.py", "w") as f:
    f.writelines(new_lines)

import py_compile
try:
    py_compile.compile("train.py", doraise=True)
    print("✅ 语法正确，开始训练")
except py_compile.PyCompileError as e:
    print(f"❌ 还有错: {e}")

✅ 语法正确，开始训练


In [27]:
import os
os.chdir("/home/ma-user/work/thruster_project")
!python -u train.py

/home/ma-user/anaconda3/envs/PyTorch-2.7.1/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/latest owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/home/ma-user/anaconda3/envs/PyTorch-2.7.1/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/8.3.RC1/aarch64-linux/ascend_toolkit_install.info owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/home/ma-user/anaconda3/envs/PyTorch-2.7.1/lib/python3.11/site-packages/torch_npu/utils/collect_env.py:58: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/latest owner does not match the current owner.
  warnings.warn(f"Warning: The {path} owner does not match the current owner.")
/home/ma-user/anaconda3/envs/PyTorch-2.7.1/lib/python3.11/site-packages/torch_